### Predicting forest Trait-Environment Misalignment under climate change

This Jupyter notebook explores how functional composition affects forest vulnerability to climate change and identifies communities where projected climate conditions create the strongest misalignment between current and future community-level functional composition.

We build a model to predict the community-weighted trait means (CWMs) of North American forests as a function of the abiotic conditions present there. We then use 2080-2100 climate projections of these same abiotic variables to predict the CWMs in the same locations to identify the traits forecast to shift the most under climate change. We then identify which existing forest groups' traits are predicted to shift more than 3 times the Standard Deviation of that trait's existing variability and flag these as misaligned.

### Dataset Description

We use forest inventory taken separately from the US and Canadian Forest Inventory and Analysis Programs totalling circa 40,000 plots. 18 physiological and morphological traits were taken from Maynard et al., representing leaf economics, wood structure, moisture regulation, belowground allocation, and tree size. Six trait syndromes - cold tolerance, shade tolerance, drought tolerance, waterlogging tolerance, and fire tolerance - were taken from Rueda et al. We extracted information of mycorrhizal symbiosis type from Averill et al.  See the separate R script - data_processing.R for more information.

### Steps
1. **Set-up**: Import the libraries and functions needed to conduct the analysis, load the datasets
2. **Model construction**: Pre-process the data scaling and splitting it
3. **CWM predictions**: Predict the change in CWMs
4. **Trait-environment misalignment (TEM)**: Calculate the TEM of each plot, trait and group
5. **Environmental covariates**: Run SHAP analysis to identify which covariates are most correlated with increases in TEM
6. **Plotting exports**: Code used to export data for plotting in R

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")
%cd ..

### 1. Set-up <br>
**Pre-process and merge dfs**

Read in the forest inventory data containing the CWMs and the plot-level abiotic information. Read in the climate projections for the same locations. Standardise layout and format

In [ ]:
import numpy as np
import pandas as pd

# Function to rename CHELSA_bio columns consistently
def rename_chelsa_columns(df):
    def clean_chelsa_name(col):
        if col.startswith('CHELSA_bio'):
            parts = col.split('_')
            if len(parts) > 1 and parts[1].startswith('bio'):
                bio_number = parts[1][3:]  # Extract number part from 'bio<number>'
                return f'CHELSA_bio{bio_number}'
        return col

    # Apply the renaming function to all columns
    df.columns = [clean_chelsa_name(col) for col in df.columns]
    return df

# Function to get a consistent numerical order for CHELSA_bio columns
def get_consistent_chelsa_order(*dfs):
    # Collect all CHELSA_bio columns from all DataFrames
    all_chelsa_columns = set()
    for df in dfs:
        chelsa_columns = [col for col in df.columns if col.startswith('CHELSA_bio')]
        all_chelsa_columns.update(chelsa_columns)
    
    # Sort the collected columns numerically
    sorted_chelsa_columns = sorted(all_chelsa_columns, key=lambda x: int(x.replace('CHELSA_bio', '')))
    return sorted_chelsa_columns

# Function to reorder columns based on a given order
def reorder_columns(df, desired_order):
    chelsa_columns = [col for col in df.columns if col.startswith('CHELSA_bio')]
    non_chelsa_columns = [col for col in df.columns if not col.startswith('CHELSA_bio')]
    
    # Ensure that only the columns that exist in the DataFrame are reordered
    chelsa_columns = [col for col in desired_order if col in chelsa_columns]
    
    # Reorder columns
    ordered_columns = non_chelsa_columns + chelsa_columns
    return df[ordered_columns]

# Function to move 'lat' and 'lon' to specific positions
def move_lat_lon_columns(df, lat_pos=None, lon_pos=None):
    columns = df.columns.tolist()  # Get the list of columns
    if 'lat' in columns and lat_pos is not None:
        columns.insert(lat_pos, columns.pop(columns.index('lat')))  # Move 'lat' to the specified position
    if 'lon' in columns and lon_pos is not None:
        columns.insert(lon_pos, columns.pop(columns.index('lon')))  # Move 'lon' to the specified position
    return df[columns]  # Reorder DataFrame

# Load and rename columns for all DataFrames
current_df = pd.read_csv('data/precomputed/plot_and_abiotic_data_current.csv')
current_df = rename_chelsa_columns(current_df)

future_climate_ssp126 = pd.read_csv('data/precomputed/plot_and_abiotic_data_ssp126.csv')
future_climate_ssp126 = rename_chelsa_columns(future_climate_ssp126)

future_climate_ssp370 = pd.read_csv('data/precomputed/plot_and_abiotic_data_ssp370.csv')
future_climate_ssp370 = rename_chelsa_columns(future_climate_ssp370)

future_climate_ssp585 = pd.read_csv('data/precomputed/plot_and_abiotic_data_ssp585.csv')
future_climate_ssp585 = rename_chelsa_columns(future_climate_ssp585)
future_climate_ssp585 = rename_chelsa_columns(future_climate_ssp585)

# Determine the consistent CHELSA_bio column order
desired_order = get_consistent_chelsa_order(current_df, future_climate_ssp126, future_climate_ssp370, future_climate_ssp585)

# Apply the same column order to all DataFrames
current_df = reorder_columns(current_df, desired_order)
future_climate_ssp126 = reorder_columns(future_climate_ssp126, desired_order)
future_climate_ssp370 = reorder_columns(future_climate_ssp370, desired_order)
future_climate_ssp585 = reorder_columns(future_climate_ssp585, desired_order)

# Move 'lat' and 'lon' columns to specific positions
current_df = move_lat_lon_columns(current_df, lat_pos=26, lon_pos=27)
future_climate_ssp126 = move_lat_lon_columns(future_climate_ssp126, lat_pos=1, lon_pos=2)  
future_climate_ssp370 = move_lat_lon_columns(future_climate_ssp370, lat_pos=1, lon_pos=2)  
future_climate_ssp585 = move_lat_lon_columns(future_climate_ssp585, lat_pos=1, lon_pos=2)  

current_df

#### 2. Model construction <br>

Split data into train/test. Train a Random Forest Regressor in a multioutput wrapper to predict the CWMs from the abiotic variables. Evaluate model performance.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
import matplotlib.pyplot as plt

trait_names = current_df.columns[1:25].tolist()

# Extract plot/community traits as targets for the models
targets = current_df.iloc[:, 1:len(trait_names) + 1]

n_traits = len(trait_names)

# Extract environmental variables as features for the models
climate_features = ['EarthEnvTopoMed_Eastness', 'EarthEnvTopoMed_Elevation',
                    'EarthEnvTopoMed_Northness', 'EarthEnvTopoMed_Slope',
                    'SG_Bulk_density_015cm', 'SG_Clay_Content_015cm',
                    'SG_Coarse_fragments_015cm', 'SG_Depth_to_bedrock',
                    'SG_Sand_Content_015cm', 'SG_Silt_Content_015cm', 'CHELSA_bio1',
                    'CHELSA_bio7', 'CHELSA_bio9', 'CHELSA_bio11', 'CHELSA_bio12',
                    'CHELSA_bio15', 'CHELSA_bio17', 'CHELSA_bio19']
features = current_df[climate_features].values

# Extract plot/community traits as targets for the models
targets = current_df.iloc[:, 1:len(trait_names) + 1]

n_traits = len(trait_names)

# Split data into train and test sets, including FIA_group information
train_indices, test_indices, features_train, features_test, target_train, target_test = train_test_split(
    current_df.index, features, targets, test_size=0.2, random_state=42, stratify=current_df.FIA_group
)

# Initialise the StandardScaler
scaler_features = StandardScaler()
scaler_targets = StandardScaler()

# Fit and transform the training data
features_train_scaled = scaler_features.fit_transform(features_train)
target_train_scaled = scaler_targets.fit_transform(target_train)

# Transform the test data using the same scaler
features_test_scaled = scaler_features.transform(features_test)
target_test_scaled = scaler_targets.transform(target_test)

# Initialise Random Forest regressor
rf_regressor = RandomForestRegressor(n_estimators=1800, random_state=42, n_jobs=-1, min_samples_split=2, min_samples_leaf=1, max_features='log2', max_depth=20, bootstrap=False)

# Wrap the Random Forest regressor with MultiOutputRegressor
multioutput_rf = MultiOutputRegressor(rf_regressor)

# Train the MultiOutput regressor model
multioutput_rf.fit(features_train_scaled, target_train_scaled)

# Predictions on test set using MultiOutput regressor
rf_pred_scaled = multioutput_rf.predict(features_test_scaled)

# Compute evaluation metrics for Random Forest 
rf_rmse_per_trait_scaled = np.sqrt(mean_squared_error(target_test_scaled, rf_pred_scaled, multioutput='raw_values'))
rf_r2_per_trait_scaled = np.array([r2_score(target_test_scaled[:, i], rf_pred_scaled[:, i]) for i in range(n_traits)])
rf_normalized_rmse_per_trait_scaled = rf_rmse_per_trait_scaled / (np.max(target_test_scaled, axis=0) - np.min(target_test_scaled, axis=0))

# Store evaluation metrics in DataFrames with trait names as index
evaluation_metrics_rf_scaled = pd.DataFrame({
    'R-squared (RF)': rf_r2_per_trait_scaled,
    'Normalised RMSE (RF)': rf_normalized_rmse_per_trait_scaled
}, index=trait_names[:n_traits])

# Calculate overall R2
overall_r2_scaled = r2_score(target_test_scaled.flatten(), rf_pred_scaled.flatten())

# Calculate overall Normalised RMSE
overall_rmse_scaled = np.sqrt(mean_squared_error(target_test_scaled.flatten(), rf_pred_scaled.flatten()))
overall_normalized_rmse_scaled = overall_rmse_scaled / (np.max(target_test_scaled) - np.min(target_test_scaled))

# Print evaluation results
print("\nRandom Forest Evaluation Metrics:")
print(evaluation_metrics_rf_scaled)

print("\nOverall Dataset Evaluation Metrics:")
print(f"Overall R-squared (RF): {overall_r2_scaled}")
print(f"Overall Normalised RMSE (RF): {overall_normalized_rmse_scaled}")

def clean_trait_name(name):
    # Clean up trait names
    cleaned_name = name.replace('_', ' ').title()
    return cleaned_name

def trait_comparisons(y_true, y_pred, trait_names):
    num_traits = y_true.shape[1]
    num_plots = num_traits
    rows = (num_plots + 2) // 3  
    cols = min(num_plots, 4)     

    fig, axs = plt.subplots(rows, cols, figsize=(24, 12*rows))
    axs = axs.flatten() 

    for i in range(num_traits):
        ax = axs[i]
        true_values = y_true[:, i]
        pred_values = y_pred[:, i]

        # Calculate R2 value
        r_squared = r2_score(true_values, pred_values)

        # Scatter plot
        ax.scatter(true_values, pred_values, color='blue', alpha=0.7)

        # Plot diagonal line 
        ax.plot([min(true_values), max(true_values)], [min(true_values), max(true_values)], color='red', linestyle='--')

        # Clean up trait name
        cleaned_name = clean_trait_name(trait_names[i])

        ax.set_title(f"{cleaned_name}\n(R-squared: {r_squared:.2f})", fontsize=16, fontweight='bold')
        ax.set_ylabel('Predicted', fontsize=12)
        ax.set_xlabel('True', fontsize=12)
        ax.set_aspect('equal')

    for j in range(num_traits, len(axs)):
        fig.delaxes(axs[j])

    fig.subplots_adjust(hspace=0.005) 
    plt.show()

# plot the R2 graphs for each trait 
trait_comparisons(target_test_scaled, rf_pred_scaled, trait_names)

predictions_df = pd.DataFrame()

# Add true and predicted values for each trait to the df
for i, trait in enumerate(trait_names[:n_traits]):
    predictions_df[f'{trait}_pred'] = rf_pred_scaled[:, i]  
    predictions_df[f'{trait}_true'] = target_test_scaled[:, i]  

# Reset the index
predictions_df.reset_index(drop=True, inplace=True)


In [ ]:
# Plot feature importances for the environmental predictors from the fitted trait-environment model

predictor_titles = {
    'EarthEnvTopoMed_Eastness': 'Eastness',
    'EarthEnvTopoMed_Elevation': 'Elevation',
    'EarthEnvTopoMed_Northness': 'Northness',
    'EarthEnvTopoMed_Slope': 'Slope',
    'SG_Bulk_density_015cm': 'Soil bulk density',
    'SG_Clay_Content_015cm': 'Soil clay',
    'SG_Coarse_fragments_015cm': 'Soil coarse fragments',
    'SG_Depth_to_bedrock': 'Depth to bedrock',
    'SG_Sand_Content_015cm': 'Soil sand',
    'SG_Silt_Content_015cm': 'Soil silt',
    'CHELSA_bio1': 'Mean temperature',
    'CHELSA_bio7': 'Temperature range',
    'CHELSA_bio9': 'Mean temperature driest quarter',
    'CHELSA_bio11': 'Temperature coldest quarter',
    'CHELSA_bio12': 'Precipitation',
    'CHELSA_bio15': 'Precipitation seasonality',
    'CHELSA_bio17': 'Precipitation driest quarter',
    'CHELSA_bio19': 'Precipitation coldest quarter'
}

rf_feature_importances = pd.DataFrame(
    {
        trait: estimator.feature_importances_
        for trait, estimator in zip(trait_names[:n_traits], multioutput_rf.estimators_)
    },
    index=climate_features
)

rf_feature_importance_summary = (
    rf_feature_importances
    .mean(axis=1)
    .sort_values(ascending=True)
)

predictor_labels = [
    predictor_titles[name]
    for name in rf_feature_importance_summary.index
]

plt.figure(figsize=(10, 8))
plt.barh(
    predictor_labels,
    rf_feature_importance_summary.values
)

plt.xlabel("Mean feature importance across traits", fontsize=12)
plt.ylabel("Predictor", fontsize=12)
plt.title("Random Forest Predictor Importance", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

#### 2. CWM predictions

Use trained model to predict CWMs under three SSPs - 126, 370 and 585. Calculate and display the change in each CWM to indentify those with the greatest predicted change in value

In [ ]:
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from pathlib import Path


# Initialise preprocessing tools
scaler_features = StandardScaler()  # For scaling features
scaler_targets = StandardScaler()  # For scaling trait values

def preprocess_data(features_df, trait_names):
    # Extract, scale features
    features = features_df[climate_features].values
    features_scaled = scaler_features.fit_transform(features)

    # Scale current trait values
    traits = features_df[trait_names].values
    traits_scaled = scaler_targets.fit_transform(traits)
    
    return features_scaled, traits_scaled

def preprocess_features(features_df):
    # Extract features, scale them
    features = features_df[climate_features].values
    features_scaled = scaler_features.transform(features)
    return features_scaled

def predict_traits(future_df, model):
    # Prepare future climate data
    features_scaled = preprocess_features(future_df)
    predicted_traits = multioutput_rf.predict(features_scaled)
    return pd.DataFrame(predicted_traits, columns=trait_names, index=future_df['pid'])

def add_metadata(predicted_df, source_df, ssp_label):
    # Merge with metadata and add SSP label
    return predicted_df.merge(source_df[['pid', 'lat', 'lon', 'FIA_group', 'ECO_NAME', 'BIOME']], on='pid').assign(SSP=ssp_label)

def compute_changes(current_df, future_df, trait_names):
    # Merge and compute changes
    merged_df = pd.merge(current_df, future_df, left_index=True, right_index=True, suffixes=('_current', '_future'))
    for trait in trait_names:
        merged_df[f'{trait}_change'] = merged_df[f'{trait}_future'] - merged_df[f'{trait}_current']
    return merged_df

def summarize_changes(changes_df, trait_names):
    # Summarize mean changes
    summary = {trait: {'mean_change': changes_df[f'{trait}_change'].mean()} for trait in trait_names}
    return pd.DataFrame(summary).T

def plot_comparative_histograms(summary_ssp126, summary_ssp370, summary_ssp585):
    fig, ax = plt.subplots(figsize=(12, 8), dpi=300)  # Set DPI to 300

    traits = summary_ssp126.index
    bar_width = 0.25
    index = range(len(traits))

    # Apply clean_trait_name function for x-axis labels
    clean_labels = [clean_trait_name(trait) for trait in traits]

    # Define color maps for SSPs
    cmap_126 = cm.get_cmap('Greens')
    cmap_370 = cm.get_cmap('Blues')  
    cmap_585 = cm.get_cmap('Oranges')
    color_126 = mcolors.to_hex(cmap_126(0.6)) 
    color_370 = mcolors.to_hex(cmap_370(0.6)) 
    color_585 = mcolors.to_hex(cmap_585(0.6))  

    # Plot bars with subtle, consistent colours
    ax.bar(index, summary_ssp126['mean_change'], bar_width, label='SSP126', color=color_126, alpha=0.8)
    ax.bar([i + bar_width for i in index], summary_ssp370['mean_change'], bar_width, label='SSP370', color=color_370, alpha=0.8)
    ax.bar([i + 2 * bar_width for i in index], summary_ssp585['mean_change'], bar_width, label='SSP585', color=color_585, alpha=0.8)

    # Set labels and title with font size
    ax.set_xlabel('Traits', fontsize=10)
    ax.set_ylabel('Mean Change', fontsize=10)
    ax.set_title('Comparative Histograms of Trait Changes Across SSP Scenarios', fontsize=10)

    # Adjust x-ticks to use cleaned trait names
    ax.set_xticks([i + bar_width for i in index])
    ax.set_xticklabels(clean_labels, rotation=45, ha='right', fontsize=10)

    # Adjust y-ticks
    ax.tick_params(axis='y', labelsize=10)

    ax.legend(fontsize=10)

    plt.tight_layout()
    plt.show()


# Preprocess current data
current_features_scaled, current_traits_scaled = preprocess_data(current_df, trait_names)
current_traits = pd.DataFrame(current_traits_scaled, columns=trait_names, index=current_df.index).join(current_df[['pid', 'lat', 'lon', 'FIA_group', 'ECO_NAME', 'BIOME']])
current_traits['SSP'] = 'current'
current_traits.to_csv("data/precomputed/current_traits_scaled.csv", index=False)

pd.DataFrame({
    "trait": trait_names,
    "mean": scaler_targets.mean_,
    "scale": scaler_targets.scale_
}).to_csv(
    "data/precomputed/target_trait_scaler_params.csv",
    index=False
)

pd.DataFrame({
    "feature": climate_features,
    "mean": scaler_features.mean_,
    "scale": scaler_features.scale_
}).to_csv(
    "data/precomputed/feature_scaler_params.csv",
    index=False
)

# Predict future traits for each SSP
predicted_traits_current = predict_traits(current_df, rf_regressor)
predicted_traits_ssp126 = predict_traits(future_climate_ssp126, rf_regressor)
predicted_traits_ssp370 = predict_traits(future_climate_ssp370, rf_regressor)
predicted_traits_ssp585 = predict_traits(future_climate_ssp585, rf_regressor)

# Add metadata and SSP labels to predicted traits
predicted_traits_current = add_metadata(predicted_traits_current, current_df, 'current')
predicted_traits_ssp126 = add_metadata(predicted_traits_ssp126, current_df, 'SSP126')
predicted_traits_ssp370 = add_metadata(predicted_traits_ssp370, current_df, 'SSP370')
predicted_traits_ssp585 = add_metadata(predicted_traits_ssp585, current_df, 'SSP585')

# Combine all DataFrames into one consolidated DataFrame
consolidated_df = pd.concat([current_traits, predicted_traits_ssp126, predicted_traits_ssp370, predicted_traits_ssp585], ignore_index=True)

# Extract the scaled current trait values
scaled_current_df = consolidated_df[consolidated_df['SSP'] == 'current'].set_index('pid')[trait_names]

# Compute changes for each SSP
changes_ssp126 = compute_changes(scaled_current_df, consolidated_df[consolidated_df['SSP'] == 'SSP126'].set_index('pid'), trait_names)
changes_ssp370 = compute_changes(scaled_current_df, consolidated_df[consolidated_df['SSP'] == 'SSP370'].set_index('pid'), trait_names)
changes_ssp585 = compute_changes(scaled_current_df, consolidated_df[consolidated_df['SSP'] == 'SSP585'].set_index('pid'), trait_names)

# Summarize changes for each SSP
summary_ssp126 = summarize_changes(changes_ssp126, trait_names)
summary_ssp370 = summarize_changes(changes_ssp370, trait_names)
summary_ssp585 = summarize_changes(changes_ssp585, trait_names)

# Plot the histograms with subtler colors
plot_comparative_histograms(summary_ssp126, summary_ssp370, summary_ssp585)

# Rename the 'mean_change' column to reflect the specific SSP
summary_ssp126 = summary_ssp126.rename(columns={'mean_change': 'SSP126'})
summary_ssp370 = summary_ssp370.rename(columns={'mean_change': 'SSP370'})
summary_ssp585 = summary_ssp585.rename(columns={'mean_change': 'SSP585'})

# Merge the DataFrames on the index (trait names)
summary_all_ssps = summary_ssp126.join(summary_ssp370).join(summary_ssp585)

# Reset index to have trait as a column instead of an index
summary_all_ssps.reset_index(inplace=True)
summary_all_ssps.rename(columns={'index': 'Trait'}, inplace=True)

# View the final DataFrame
print(summary_all_ssps)

summary_all_ssps.to_csv("output/all_trait_changes.csv")

# save summary DataFrames
summary_ssp126.to_csv("output/summary_trait_change_ssp126.csv")
summary_ssp370.to_csv("output/summary_trait_change_ssp370.csv")
summary_ssp585.to_csv("output/summary_trait_change_ssp585.csv")

# save absolute trait change for SSP370 plotting
out_path = Path("data/precomputed/changes_ssp370_abs_scale_trait_chg.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)

change_cols = [c for c in changes_ssp370.columns if c.endswith("_change")]

changes_ssp370_abs_scale_trait_chg = changes_ssp370[["lat", "lon"]].copy()
changes_ssp370_abs_scale_trait_chg["pid"] = changes_ssp370.index.astype(str)

changes_ssp370_abs_scale_trait_chg["abs_scaled_change"] = (
    changes_ssp370[change_cols].abs().sum(axis=1)
)

changes_ssp370_abs_scale_trait_chg = changes_ssp370_abs_scale_trait_chg[
    ["pid", "lat", "lon", "abs_scaled_change"]
]

changes_ssp370_abs_scale_trait_chg.to_csv(out_path, index=False)

print(f"Saved: {out_path}")

**Test for statistical significance of predicted trait changes vs current values**

In [ ]:
from scipy import stats
from statsmodels.stats.multitest import multipletests
from scipy.stats import chi2_contingency, wilcoxon

# Function to perform Wilcoxon Signed-Rank Tests comparing each SSP
def perform_wilcoxon_tests(changes_ssp126, changes_ssp370, changes_ssp585, trait_names):
    wilcoxon_results = {}

    for trait in trait_names:
        # Get the current and future trait values for each SSP
        ssp126_changes = changes_ssp126[[f'{trait}_current', f'{trait}_future']].dropna()
        ssp370_changes = changes_ssp370[[f'{trait}_current', f'{trait}_future']].dropna()
        ssp585_changes = changes_ssp585[[f'{trait}_current', f'{trait}_future']].dropna()
        
        # Align the data based on index (pid)
        merged_ssp126 = ssp126_changes.join(ssp370_changes[f'{trait}_future'], how='inner', rsuffix='_370')
        merged_ssp126 = merged_ssp126.join(ssp585_changes[f'{trait}_future'], how='inner', rsuffix='_585')

        # Perform Wilcoxon Signed-Rank test for SSP126 vs Current
        current_aligned = merged_ssp126[f'{trait}_current']
        ssp126_aligned = merged_ssp126[f'{trait}_future']
        if len(ssp126_aligned) > 0 and len(current_aligned) > 0:
            try:
                w_stat, p_val = stats.wilcoxon(current_aligned, ssp126_aligned)
                wilcoxon_results[f'SSP126_vs_Current_{trait}'] = {
                    'W-statistic': w_stat, 
                    'p-value': p_val
                }
            except ValueError as e:
                wilcoxon_results[f'SSP126_vs_Current_{trait}'] = {
                    'W-statistic': np.nan, 
                    'p-value': np.nan,
                    'Error': str(e)
                }

        # Repeat for SSP370
        merged_ssp370 = ssp126_changes.join(ssp370_changes[f'{trait}_future'], how='inner', rsuffix='_370')

        current_aligned = merged_ssp370[f'{trait}_current']
        ssp370_aligned = merged_ssp370[f'{trait}_future_370']
        if len(ssp370_aligned) > 0 and len(current_aligned) > 0:
            try:
                w_stat, p_val = stats.wilcoxon(current_aligned, ssp370_aligned)
                wilcoxon_results[f'SSP370_vs_Current_{trait}'] = {
                    'W-statistic': w_stat, 
                    'p-value': p_val
                }
            except ValueError as e:
                wilcoxon_results[f'SSP370_vs_Current_{trait}'] = {
                    'W-statistic': np.nan, 
                    'p-value': np.nan,
                    'Error': str(e)
                }

        # Repeat for SSP585
        merged_ssp585 = ssp126_changes.join(ssp585_changes[f'{trait}_future'], how='inner', rsuffix='_585')

        current_aligned = merged_ssp585[f'{trait}_current']
        ssp585_aligned = merged_ssp585[f'{trait}_future_585']
        if len(ssp585_aligned) > 0 and len(current_aligned) > 0:
            try:
                w_stat, p_val = stats.wilcoxon(current_aligned, ssp585_aligned)
                wilcoxon_results[f'SSP585_vs_Current_{trait}'] = {
                    'W-statistic': w_stat, 
                    'p-value': p_val
                }
            except ValueError as e:
                wilcoxon_results[f'SSP585_vs_Current_{trait}'] = {
                    'W-statistic': np.nan, 
                    'p-value': np.nan,
                    'Error': str(e)
                }

    # Convert results to DataFrame
    results_df = pd.DataFrame(wilcoxon_results).T
    p_values = results_df['p-value'].dropna()
    
    if len(p_values) > 0:
        # Apply Bonferroni correction
        adjusted_pvals = multipletests(p_values, method='bonferroni')[1]
        results_df.loc[p_values.index, 'Adjusted p-value'] = adjusted_pvals
    else:
        # If there are no valid p-values, set the adjusted column to NaN
        results_df['Adjusted p-value'] = np.nan

    # Determine significance after correction
    results_df['Statistical significance'] = results_df['Adjusted p-value'].apply(lambda p: 'Yes' if p < 0.05 else 'No')
    
    return results_df

# Function to visualise Wilcoxon test results
def plot_wilcoxon_results_vs_current(wilcoxon_results):
    comparisons = wilcoxon_results.index
    w_stats = wilcoxon_results['W-statistic']
    p_vals = wilcoxon_results['p-value']

    fig, ax1 = plt.subplots(figsize=(14, 7))

    # Plot W-statistics on primary y-axis
    ax1.bar(comparisons, w_stats, color='skyblue', edgecolor='black', label='W-statistic')
    ax1.set_xlabel('Comparison')
    ax1.set_ylabel('W-statistic', color='skyblue')
    ax1.tick_params(axis='y', labelcolor='skyblue')
    ax1.set_xticklabels(comparisons, rotation=90)
    
    # Create secondary y-axis to plot p-values
    ax2 = ax1.twinx()
    ax2.plot(comparisons, p_vals, color='salmon', marker='o', linestyle='--', label='p-value')
    ax2.set_ylabel('p-value', color='salmon')
    ax2.tick_params(axis='y', labelcolor='salmon')
    ax2.set_yscale('log')  # Log scale for p-values

    # Add legends and title
    ax1.legend(loc='upper left')
    ax2.legend(loc='upper right')
    plt.title('Wilcoxon Signed-Rank Test Results Comparing SSPs vs. Current')
    
    plt.tight_layout()
    plt.show()

# Function to run the tests and visualise the results
def run_statistical_tests_and_visualize(changes_ssp126, changes_ssp370, changes_ssp585, trait_names):
    # Perform Wilcoxon Signed-Rank Tests comparing SSPs vs. Current data
    wilcoxon_results_vs_current = perform_wilcoxon_tests(changes_ssp126, changes_ssp370, changes_ssp585, trait_names)
    
    # Print the summary table of results
    print("\nSummary of Wilcoxon Signed-Rank Tests comparing SSPs vs. Current:")
    print(wilcoxon_results_vs_current)

    # Visualize Wilcoxon test results
    plot_wilcoxon_results_vs_current(wilcoxon_results_vs_current)

    # Return the summary table of results
    return wilcoxon_results_vs_current

# Run the tests and get the summary table of results
summary_table = run_statistical_tests_and_visualize(changes_ssp126, changes_ssp370, changes_ssp585, trait_names)

# Display the returned summary table if needed
print(summary_table)

summary_table.to_csv("output/wilcoxon_test_results.csv")

#### 4. Trait-Environment Misalignment (TEM)

Calculate the minimum Mahalanobis distance (MD) between the future plot traits and any of the current plot traits including perturbed plot traits with sampled variation. Separately calculate the MD from the future plot traits to the baseline (unperturbed) current traits.

In [ ]:
import pyarrow.parquet as pq
import glob

MC_DIR = Path("data/precomputed/forest_trait_means_mc/")
MC_PATTERN = "forest_trait_means_mc_*.parquet"

# Include dropped traits as fixed (non-perturbed) dimensions in the distance
INCLUDE_FIXED_DROPPED_TRAITS = True

# Dropped traits (syndromes + myco)
DROPPED_TRAITS = ["shade", "drought", "cold", "water", "fire_tol", "myco_association"]

ASSUME_FIXED_FUTURE_EQUALS_CURRENT_IF_MISSING = True

# Exact quantiles require storing all distances (R x N) in memory
STORE_ALL_DISTANCES_FOR_QUANTILES = True

# Guard for tiny SDs in normalisation
MIN_SD = 1e-12

# Regularisation for covariance inversion 
COV_RIDGE = 1e-6

# Filter FIA groups
group_counts = changes_ssp370["FIA_group"].value_counts()
valid_groups = group_counts[group_counts > 30].index
valid_groups = [group for group in valid_groups if "Other" not in group]

df = changes_ssp370.loc[changes_ssp370["FIA_group"].isin(valid_groups)].copy()
df["pid"] = df.index.astype(str)

# Load replicate file list + read first replicate to define intersection
rep_files = sorted(glob.glob(str(MC_DIR / MC_PATTERN)))
if len(rep_files) == 0:
    raise FileNotFoundError(f"No replicate parquet files found in {MC_DIR} matching {MC_PATTERN}")

rep0 = pq.read_table(rep_files[0]).to_pandas()
rep0["pid"] = rep0["pid"].astype(str)

rep_pid_set = set(rep0["pid"].tolist())

before_n = len(df)
df = df.loc[df["pid"].isin(rep_pid_set)].copy()
after_n = len(df)

dropped_n = before_n - after_n
if dropped_n > 0:
    print(f"Dropping {dropped_n} plots not found in MC parquets; continuing with {after_n} plots.")

# Stable pid order to align everything
pid_order = df["pid"].to_list()
n_plots = len(pid_order)

# Traits present in parquet replicate outputs
parquet_trait_cols = [c for c in rep0.columns if c not in ("pid", "replicate")]

perturbed_traits = [
    t for t in trait_names
    if (t in parquet_trait_cols) and (f"{t}_current" in df.columns) and (f"{t}_future" in df.columns)
]

# add fixed (dropped) traits (from df columns)
fixed_traits = []
if INCLUDE_FIXED_DROPPED_TRAITS:
    for t in DROPPED_TRAITS:
        cur_col = f"{t}_current"
        fut_col = f"{t}_future"

        if cur_col in df.columns and fut_col in df.columns:
            fixed_traits.append(t)

        elif cur_col in df.columns and fut_col not in df.columns and ASSUME_FIXED_FUTURE_EQUALS_CURRENT_IF_MISSING:
            df[fut_col] = df[cur_col]
            fixed_traits.append(t)

dist_traits = perturbed_traits + fixed_traits
if len(dist_traits) == 0:
    raise ValueError("No traits available for distance calculation after applying filters/toggles.")

# Anchored SDs of current traits by FIA_group
trait_columns_current = [f"{t}_current" for t in dist_traits]
trait_columns_future  = [f"{t}_future" for t in dist_traits]

group_trait_sds = df.groupby("FIA_group")[trait_columns_current].std()

# Attach SDs to each row via FIA_group join 
sds_df = group_trait_sds.reset_index()
sds_df.columns = ["FIA_group"] + [f"sd__{t}" for t in dist_traits]
df = df.merge(sds_df, on="FIA_group", how="left")

# Re-create pid_order after merge
pid_order = df["pid"].to_list()
n_plots = len(pid_order)

sd_cols = [f"sd__{t}" for t in dist_traits]
sd_mat = df[sd_cols].to_numpy(dtype=float)
sd_mat = np.where(sd_mat < MIN_SD, MIN_SD, sd_mat)

# Normalise future once
future_mat = df[trait_columns_future].to_numpy(dtype=float)
future_norm = future_mat / sd_mat

# Mahalanobis prep: global inverse covariance in the normalised space
current_base_mat = df[trait_columns_current].to_numpy(dtype=float)
current_base_norm = current_base_mat / sd_mat

p = len(dist_traits)
I_p = np.eye(p, dtype=float)

cov_global = np.cov(current_base_norm, rowvar=False, bias=False)
cov_global = cov_global + (COV_RIDGE * I_p)
VI_global = np.linalg.pinv(cov_global)

# Loop replicates: compute distances, summarise per plot and store per-trait contributions for the min replicate
R = len(rep_files)

mahal_min = np.full(n_plots, np.inf, dtype=np.float64)
sum_d = np.zeros(n_plots, dtype=np.float64)
sum_d2 = np.zeros(n_plots, dtype=np.float64)

if STORE_ALL_DISTANCES_FOR_QUANTILES:
    d_all = np.empty((R, n_plots), dtype=np.float32)

# Pre-extract fixed-current matrix
fixed_current_mat = None
if len(fixed_traits) > 0:
    fixed_current_mat = df[[f"{t}_current" for t in fixed_traits]].to_numpy(dtype=float)

# Store per-trait contribution to min distance for each plot
p = len(dist_traits)
min_contrib_sq = np.zeros((n_plots, p), dtype=np.float32)

for i, f in enumerate(rep_files):
    rep = pq.read_table(f).to_pandas()
    rep["pid"] = rep["pid"].astype(str)

    # Align replicate rows to pid_order
    rep = rep.set_index("pid").loc[pid_order]

    parts = []
    if len(perturbed_traits) > 0:
        parts.append(rep[perturbed_traits].to_numpy(dtype=float))

    if fixed_current_mat is not None:
        parts.append(fixed_current_mat)

    current_mat = np.column_stack(parts)  # columns match dist_traits
    current_norm = current_mat / sd_mat

    delta = future_norm - current_norm

    # Compute VI_global @ delta for each row
    # v has shape (n_plots, p)
    v = delta @ VI_global

    # Quadratic form per row
    q = np.einsum("ij,ij->i", delta, v)
    q = np.maximum(q, 0.0)
    d = np.sqrt(q)

    # Update running summaries across replicates
    mahal_min = np.minimum(mahal_min, d)
    sum_d += d
    sum_d2 += d * d

    if STORE_ALL_DISTANCES_FOR_QUANTILES:
        d_all[i, :] = d.astype(np.float32)

    # Calculate per-trait misalignment
    contrib_sq = delta * delta  # shape (n_plots, p), always >= 0

    if i == 0:
        prev_min = np.full(n_plots, np.inf, dtype=np.float64)

    strict_improved = d < prev_min

    if np.any(strict_improved):
        min_contrib_sq[strict_improved, :] = contrib_sq[strict_improved, :].astype(np.float32)

    prev_min = np.minimum(prev_min, d)

mahal_mean = sum_d / max(R, 1)
mahal_var = (sum_d2 / max(R, 1)) - (mahal_mean ** 2)
mahal_sd = np.sqrt(np.maximum(mahal_var, 0.0))

if STORE_ALL_DISTANCES_FOR_QUANTILES:
    mahal_p05 = np.quantile(d_all, 0.05, axis=0)
    mahal_p50 = np.quantile(d_all, 0.50, axis=0)
    mahal_p95 = np.quantile(d_all, 0.95, axis=0)
else:
    mahal_p05 = np.full(n_plots, np.nan)
    mahal_p50 = np.full(n_plots, np.nan)
    mahal_p95 = np.full(n_plots, np.nan)

# Convert per-trait min contributions into distance units
min_contrib = np.sqrt(np.maximum(min_contrib_sq, 0.0)).astype(np.float32)

# Output per-plot summaries for R mapping plus group x trait mean(min contribution) matrix for heatmaps
mahal_summary_df = df[["pid", "lat", "lon", "FIA_group"]].copy()

mahal_summary_df["mahal_min"] = mahal_min
mahal_summary_df["mahal_mean"] = mahal_mean
mahal_summary_df["mahal_sd"] = mahal_sd
mahal_summary_df["mahal_p05"] = mahal_p05
mahal_summary_df["mahal_p50"] = mahal_p50
mahal_summary_df["mahal_p95"] = mahal_p95

mahal_summary_df["n_reps"] = R
mahal_summary_df["n_traits_total"] = len(dist_traits)
mahal_summary_df["n_traits_perturbed"] = len(perturbed_traits)
mahal_summary_df["n_traits_fixed"] = len(fixed_traits)

mahal_summary_df.to_csv(
    "data/precomputed/mahalanobis_distance_all_mc.csv",
    index=False
)


BASELINE_PARQUET = Path(
    "data/precomputed/forest_trait_means_mc/forest_trait_means_mc_baseline.parquet"
)

BASELINE_OUT_CSV = Path(
    "data/precomputed/mahalanobis_distance_baseline_min_only.csv"
)

# Read baseline parquet
baseline_df = pq.read_table(str(BASELINE_PARQUET)).to_pandas()
baseline_df["pid"] = baseline_df["pid"].astype(str)

# Align baseline rows to pid order used in main analysis
baseline_df = baseline_df.set_index("pid").loc[pid_order]

# Build baseline current matrix using same trait order as dist_traits
baseline_parts = []

if len(perturbed_traits) > 0:
    baseline_parts.append(baseline_df[perturbed_traits].to_numpy(dtype=float))

if fixed_current_mat is not None:
    baseline_parts.append(fixed_current_mat)

baseline_current_mat = np.column_stack(baseline_parts)
baseline_current_norm = baseline_current_mat / sd_mat

# Compute Mahalanobis distance (future vs baseline current)
baseline_delta = future_norm - baseline_current_norm
baseline_v = baseline_delta @ VI_global
baseline_q = np.einsum("ij,ij->i", baseline_delta, baseline_v)
baseline_q = np.maximum(baseline_q, 0.0)

baseline_mahal_min = np.sqrt(baseline_q)

# Export CSV with only required fields
baseline_out = df[["pid", "lat", "lon"]].copy()
baseline_out["mahal_min"] = baseline_mahal_min

baseline_out.to_csv(BASELINE_OUT_CSV, index=False)

Plot the mean TEM value per trait and per group

In [ ]:
# Plot-level per-trait contributions for the min replicate
contrib_cols = [f"min_mahal_{t}" for t in dist_traits]
out_contrib = df[["pid", "FIA_group"]].copy()
out_contrib[contrib_cols] = min_contrib

out_contrib.to_csv(
    "data/precomputed/mahalanobis_min_trait_contrib_per_plot.csv",
    index=False
)

# Group x trait matrix of mean(min contribution) per trait per group
group_means = out_contrib.groupby("FIA_group")[contrib_cols].mean()

# Overall column and overall row
group_means["Overall"] = group_means.mean(axis=1)
overall_row = group_means.mean(axis=0).to_frame().T

heatmap_with_overall = pd.concat([group_means, overall_row], axis=0)

# Sort rows and columns high to low, excluding the Overall Average row and Overall column in the sort keys
rows_sorted = heatmap_with_overall.iloc[:-1, :-1].mean(axis=1).sort_values(ascending=False).index
cols_sorted = heatmap_with_overall.iloc[:-1, :-1].mean(axis=0).sort_values(ascending=False).index

heatmap_sorted = heatmap_with_overall.loc[list(rows_sorted), list(cols_sorted) + ["Overall"]]

heatmap_sorted.to_csv(
    "output/min_mahal_matrix_sorted.csv"
)

# Clean trait names for display
def clean_trait_column(trait_column: str) -> str:
    s = str(trait_column)

    # Remove common per-trait MD prefixes
    for pref in ("min_mahal_", "mahal_min_", "min_mahalanobis_", "mahalanobis_min_"):
        if s.startswith(pref):
            s = s[len(pref):]

    # If you used a suffix style instead
    for suff in ("_min_mahal", "_mahal_min", "_min_mahalanobis", "_mahalanobis_min"):
        if s.endswith(suff):
            s = s[: -len(suff)]

    return s.replace("_", " ").title()

# Clean FIA group labels for display
def clean_fia_group(fia_group: str) -> str:
    s = str(fia_group)
    s = s.replace("group", "").strip()
    return s

heatmap_plot = heatmap_sorted.copy()

# Ensure there is an Overall column
has_overall = "Overall" in heatmap_plot.columns
trait_cols = [c for c in heatmap_plot.columns if c != "Overall"]

if has_overall:
    heatmap_plot = heatmap_plot[trait_cols + ["Overall"]]

# Sort groups high to low using mean across traits
group_order = heatmap_plot[trait_cols].mean(axis=1).sort_values(ascending=False).index
heatmap_plot = heatmap_plot.loc[group_order]

# Sort traits high to low using mean across groups
trait_order = heatmap_plot[trait_cols].mean(axis=0).sort_values(ascending=False).index.tolist()
trait_cols = trait_order
if has_overall:
    heatmap_plot = heatmap_plot[trait_cols + ["Overall"]]
else:
    heatmap_plot = heatmap_plot[trait_cols]

# Create figure
fig, ax_main = plt.subplots(figsize=(14, 10), dpi=300)

# Plot heatmap for per-trait mean min values (now: |delta| in SD units)
im = ax_main.imshow(
    heatmap_plot[trait_cols].to_numpy(),
    aspect="auto",
    cmap="Blues",
    interpolation="nearest"
)

# X axis labels
ax_main.set_xticks(range(len(trait_cols)))
ax_main.set_xticklabels(
    [clean_trait_column(t) for t in trait_cols],
    rotation=45,
    ha="right",
    rotation_mode="anchor",
    fontsize=16
)

# Y axis labels
ax_main.set_yticks(range(heatmap_plot.shape[0]))
ax_main.set_yticklabels(
    [clean_fia_group(f) for f in heatmap_plot.index],
    fontsize=16
)

# Labels
ax_main.set_xlabel("Traits", fontsize=16)
ax_main.set_ylabel("Forest Groups", fontsize=16)

# Colour bar
cbar = plt.colorbar(im, ax=ax_main, fraction=0.046, pad=0.04)
cbar.set_label("Current vs Future Trait difference (group SD)", fontsize=16)
cbar.ax.tick_params(labelsize=14)

# Add Overall group values as a right-hand text column
if has_overall:
    overall_vals = heatmap_plot["Overall"].to_numpy()
    x_overall = len(trait_cols)

    for i, value in enumerate(overall_vals):
        ax_main.text(
            x_overall, i, f"{value:.2f}",
            ha="center", va="center",
            fontsize=10, color="black", fontweight="bold",
            bbox=dict(facecolor="white", edgecolor="black")
        )

    ax_main.axvline(x=len(trait_cols) - 0.5, color="black", linewidth=1.5)

# Layout
plt.tight_layout()
plt.subplots_adjust(right=1.4)

ax_main.set_title("", fontsize=16)
plt.show()

#### 5. Environmental covariates

Model the impact of different environmental, ecological and management covariates on changes in TEM using SHAP (SHapley Additive exPlanations) values

**SHAP data pre-processing**

Define the covariates we're interested in analysing. Calculate the predcited change in climate variables for 2100 under SSP370. Merge with per plot FDR values

In [ ]:
# Filter only valid groups
min_group_size = 90 # As consolidated_df contains 3x SSPs the min_group_size is 3x as large (90 rather than 30)
group_counts = consolidated_df['FIA_group'].value_counts()
valid_groups = group_counts[group_counts >= min_group_size].index
valid_groups = [group for group in valid_groups if "Other" not in group]
consolidated_df = consolidated_df[consolidated_df['FIA_group'].isin(valid_groups)]
consolidated_df_370 = consolidated_df[consolidated_df["SSP"] == 'SSP370']
consolidated_df_370

In [ ]:
# Define features and target
combined_features_biome = [
    'CHELSA_bio1', 'CHELSA_bio7', 'CHELSA_bio12', 'CHELSA_bio9', 'CHELSA_bio11','CHELSA_bio15','CHELSA_bio17', 'CHELSA_bio19',
    'EarthEnvTopoMed_Elevation', 'EarthEnvTopoMed_Slope',
    'SG_Bulk_density_015cm', 'SG_Clay_Content_015cm',
    'SG_Coarse_fragments_015cm', 'SG_Depth_to_bedrock',
    'SG_Sand_Content_015cm', 'SG_Silt_Content_015cm', 'managed', 'OWNCD' , 'STDAGE', 'num_species'
]

ssp370_df = consolidated_df_370

ssp370_df = ssp370_df.merge(
    future_climate_ssp370[['pid'] + combined_features_biome],
    on='pid',
    how='left'
)

climate_variables = ['CHELSA_bio1',
                    'CHELSA_bio7', 'CHELSA_bio9', 'CHELSA_bio11', 'CHELSA_bio12',
                    'CHELSA_bio15', 'CHELSA_bio17', 'CHELSA_bio19']


def compute_climate_changes(current_df, future_df, climate_variables):
    # Filter for the specified climate variables in both DataFrames
    current_climate = current_df[['pid'] + climate_variables]
    future_climate = future_df[['pid'] + climate_variables]
    
    # Merge filtered DataFrames
    merged_df = pd.merge(current_climate, future_climate, on='pid', suffixes=('_current', '_future'))
    
    # Compute changes only for the specified climate variables
    for climate in climate_variables:
        merged_df[f'{climate}_change'] = merged_df[f'{climate}_future'] - merged_df[f'{climate}_current']
    
    # Return DataFrame with original columns and the calculated changes
    return merged_df


climate_changes = compute_climate_changes(current_df, ssp370_df, climate_variables)

# Correctly merge only the _change columns into future_climate_ssp370
future_climate_ssp370_exp = ssp370_df.merge(
    climate_changes[['pid'] + [f'{climate}_change' for climate in climate_variables] + [f'{climate}_future' for climate in climate_variables]],
    on='pid'
)

future_climate_ssp370_exp = future_climate_ssp370_exp.merge(mahal_summary_df[['pid', 'mahal_min', 'mahal_mean']], on = 'pid', how = 'left', validate='one_to_one')

future_climate_ssp370_exp

Select the covariates, train the RFR and extract the SHAP values

In [ ]:
import shap

combined_features_biome_all = ['CHELSA_bio1', 'CHELSA_bio7', 'CHELSA_bio12',
    'CHELSA_bio11', 'CHELSA_bio15', 'CHELSA_bio17',
    'CHELSA_bio1_change', 'CHELSA_bio12_change',
    'EarthEnvTopoMed_Elevation', 'EarthEnvTopoMed_Slope','SG_Clay_Content_015cm',
    'SG_Depth_to_bedrock',
    'managed', 'OWNCD', 'STDAGE', 'num_species' 
]

target_variable = 'mahal_min'

# Dictionary for human-readable feature names
chelsa_bio_titles = {
    'CHELSA_bio1': 'Mean temperature',
    'CHELSA_bio7': 'Temperature range',
    'CHELSA_bio11': 'Temperature coldest quarter',
    'CHELSA_bio12': 'Precipitation',
    'CHELSA_bio15': 'Precipitation seasonality',
    'CHELSA_bio17': 'Precipitation driest quarter',
    'CHELSA_bio1_change': 'Temperature - change',
    'CHELSA_bio12_change': 'Precipitation - change',
    'EarthEnvTopoMed_Elevation': 'Elevation',
    'EarthEnvTopoMed_Slope': 'Slope',
    'SG_Clay_Content_015cm': 'Soil clay',
    'SG_Depth_to_bedrock': 'Depth to bedrock',
    'managed': 'Human activity',
    'OWNCD': 'Owner class code',
    'STDAGE': 'Stand Age',
    'num_species': 'Number of Species' 
}

human_readable_combined_features = [chelsa_bio_titles.get(f, f) for f in combined_features_biome_all]

# Filter for SSP370
ssp370_df = future_climate_ssp370_exp

# Split data into training and testing
X = ssp370_df[combined_features_biome_all]
y = ssp370_df[target_variable]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Random Forest Regressor
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_regressor.fit(X_train_scaled, y_train)

# Evaluate the model
y_pred = rf_regressor.predict(X_test_scaled)
print(f"R-squared: {r2_score(y_test, y_pred):.4f}")
print(f"Mean Squared Error: {mean_squared_error(y_test, y_pred):.4f}")

# Convert X_test_scaled into a df
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=combined_features_biome_all)

# SHAP analysis
explainer = shap.TreeExplainer(rf_regressor)
shap_values = explainer.shap_values(X_test_scaled)

# Calculate mean absolute SHAP values for each feature
mean_shap_values = np.abs(shap_values).mean(axis=0)

# Update feature names to include mean absolute SHAP values
feature_names_with_shap = [
    f"{human_readable_combined_features[i]} ({mean_shap_values[i]:.3f})"
    for i in range(len(human_readable_combined_features))
]
plt.rcParams.update({'font.size': 20})

# Create SHAP summary plot 
shap.summary_plot(
    shap_values, 
    X_test_scaled_df,  
    feature_names=feature_names_with_shap,  
    plot_size=(24, 12),
    max_display=len(combined_features_biome_all),
    color=cm.get_cmap('tab10'),
    show=False  
)

# Get the current plot's axis
ax = plt.gca()

# Change font size and font color of y-axis labels (feature names)
ax.tick_params(axis='y', labelsize=20, colors='black')
ax.tick_params(axis='x', labelsize=20, colors='black')

# Set x-axis label 
ax.set_xlabel("Impact on TEM risk", fontsize=20, color='black')
for collection in ax.collections:
    collection.set_alpha(0.99)  
    collection.set_sizes([5])  

plt.savefig("output/shap_summary_plot.png", dpi=600, bbox_inches='tight')

# Show the updated plot
plt.show()

In [ ]:
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe

# Toggles / appearance
colour_by_biome = True
alpha_by_density = True
add_gam_and_ci = False

point_size = 26
edge_colour = (0, 0, 0, 0.55)
edge_width = 0.35
base_alpha = 1.0
min_alpha, max_alpha = 1.0, 1.0
density_gamma = 0.75

pygam_n_splines = 8
pygam_lam_grid = np.logspace(0, 5, 16)

_rng = np.random.default_rng(42)

# Specify the features
selected_features = [
    'num_species',
    'CHELSA_bio15',
    'STDAGE',
    'CHELSA_bio1',
    'managed'
]

# Define the colour map
biome_color_map = {
    'Tundra': '#bdbdbd',
    'Boreal Forests/Taiga': '#1f78b4',
    'Deserts & Xeric Shrublands': '#e6ab02',
    'Temperate Broadleaf & Mixed Forests': '#33a02c',
    'Temperate Conifer Forests': '#6a3d9a',
    'Temperate Grasslands, Savannas & Shrublands': '#ff7f00'
}


def to_numeric_safe(s: pd.Series) -> np.ndarray:
    return pd.to_numeric(s, errors="coerce").to_numpy(dtype=float)


def density_alphas(
    x,
    y,
    min_alpha=min_alpha,
    max_alpha=max_alpha,
    gamma=density_gamma
):
    from scipy.stats import gaussian_kde

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    m = np.isfinite(x) & np.isfinite(y)

    xy = np.vstack([x[m], y[m]])
    dens = gaussian_kde(xy)(xy)

    d_lo, d_hi = np.percentile(dens, [5, 95])
    dens = np.clip(dens, d_lo, d_hi)

    dn = (dens - d_lo) / (d_hi - d_lo + 1e-12)
    dn = dn**gamma

    out = np.full(x.shape, (min_alpha + max_alpha) / 2.0, dtype=float)
    out[m] = min_alpha + dn * (max_alpha - min_alpha)

    return out


def gam_line_and_ci_pygam(
    x,
    y,
    n_splines=pygam_n_splines,
    lam_grid=pygam_lam_grid
):
    from pygam import LinearGAM, s

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    m = np.isfinite(x) & np.isfinite(y)

    x0, y0 = x[m], y[m]
    xmin, xmax = np.min(x0), np.max(x0)

    x0s = (x0 - xmin) / (xmax - xmin)
    xg = np.linspace(xmin, xmax, 400)
    xgs = (xg - xmin) / (xmax - xmin)

    gam = LinearGAM(s(0, n_splines=n_splines))
    gam.gridsearch(x0s.reshape(-1, 1), y0, lam=lam_grid)

    mean = gam.predict(xgs.reshape(-1, 1))
    ci = gam.confidence_intervals(xgs.reshape(-1, 1), width=0.95)

    return xg, mean, ci[:, 0], ci[:, 1]


valid_biomes = set(biome_color_map.keys())

common_indices = (
    ssp370_df
    .loc[ssp370_df["BIOME"].isin(valid_biomes)]
    .index
    .intersection(X_test.index)
)

X_test_filtered = X_test.loc[common_indices].copy()
ssp370_df_filtered = ssp370_df.loc[common_indices].copy()

positional_indices = [
    X_test.index.get_loc(idx)
    for idx in common_indices
]

shap_values_filtered = shap_values[positional_indices]
biome_labels_filtered = ssp370_df_filtered["BIOME"]

X_test_filtered["STDAGE"] = X_test_filtered["STDAGE"].clip(upper=500)


fig, axs = plt.subplots(
    nrows=2,
    ncols=3,
    figsize=(18, 12),
    dpi=300,
    layout="constrained"
)

fig.set_constrained_layout_pads(
    w_pad=0.08,
    h_pad=0.08,
    wspace=0.12,
    hspace=0.12
)

axs = axs.flatten()
plot_axes = axs[:-1]
legend_ax = axs[-1]


for ax, feature in zip(plot_axes, selected_features):

    try:
        ax.set_box_aspect(1)
    except Exception:
        pass

    x = to_numeric_safe(X_test_filtered[feature])

    y = np.asarray(
        shap_values_filtered[:, combined_features_biome_all.index(feature)],
        dtype=float
    )

    x_plot = x.copy()

    if feature == "managed":
        x_plot += _rng.uniform(-0.06, 0.06, size=x_plot.size)

    m = np.isfinite(x_plot) & np.isfinite(y)

    readable = chelsa_bio_titles.get(feature, feature)

    ax.set_title(readable, fontsize=18, fontweight="bold", pad=8)
    ax.set_xlabel(readable, fontsize=15)
    ax.set_ylabel("")
    ax.axhline(0, color="grey", linestyle="dashed", linewidth=1)

    if colour_by_biome:
        cols = biome_labels_filtered.map(biome_color_map).fillna("grey").to_numpy()
    else:
        cols = np.full(len(x_plot), "grey", dtype=object)

    rgba = np.array(
        [mcolors.to_rgba(c) for c in cols],
        dtype=float
    )

    if alpha_by_density:
        rgba[:, 3] = density_alphas(x_plot, y)
    else:
        rgba[:, 3] = base_alpha

    ax.scatter(
        x_plot[m],
        y[m],
        c=rgba[m],
        s=14,
        edgecolors="none"
    )

    if add_gam_and_ci and feature != "managed":
        xg, mean, lo, hi = gam_line_and_ci_pygam(x[m], y[m])

        gam_line_colour = "#111111"
        ci_colour = "#111111"

        ax.fill_between(
            xg,
            lo,
            hi,
            color=ci_colour,
            alpha=0.12,
            linewidth=0,
            zorder=2
        )

        (line,) = ax.plot(
            xg,
            mean,
            color=gam_line_colour,
            linewidth=1.5,
            zorder=6
        )

        line.set_path_effects([
            pe.Stroke(linewidth=3.5, foreground="white"),
            pe.Normal()
        ])

    if feature == "managed":
        ax.set_xticks([0, 1])
        ax.set_xticklabels(["No", "Yes"])

        if add_gam_and_ci:
            summary_colour = "#111111"

            for grp in [0, 1]:
                gmask = m & (np.asarray(x) == grp)
                yg = y[gmask]

                mu = yg.mean()
                se = yg.std(ddof=1) / np.sqrt(yg.size)
                lo, hi = mu - 1.96 * se, mu + 1.96 * se

                (ci_line,) = ax.plot(
                    [grp, grp],
                    [lo, hi],
                    color=summary_colour,
                    linewidth=2.4,
                    zorder=7
                )

                ci_line.set_path_effects([
                    pe.Stroke(linewidth=3.0, foreground="white"),
                    pe.Normal()
                ])

                (mean_pt,) = ax.plot(
                    grp,
                    mu,
                    marker="o",
                    markersize=3.5,
                    color=summary_colour,
                    linestyle="None",
                    zorder=8
                )

                mean_pt.set_path_effects([
                    pe.Stroke(linewidth=4.8, foreground="white"),
                    pe.Normal()
                ])


legend_ax.axis("off")

if colour_by_biome:
    handles = [
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color=c,
            linestyle="",
            markersize=8,
            label=b
        )
        for b, c in biome_color_map.items()
    ]

    legend_ax.legend(
        handles=handles,
        title="BIOME",
        loc="center",
        frameon=False,
        fontsize=14,
        title_fontsize=14
    )


fig.supylabel(
    "Impact on TEM risk",
    fontsize=17,
    fontweight="bold"
)

plt.show()

#### 6. Plotting data

Additional code to produce supplementary materials and/or to export data for use in R plots


**Re-run CWM change graphs using Bootstrapped results and confidence intervals** <br>

Bootstrapped results can be re-created using the separate code file - cwm_predictions_bootstrap_runs.ipynb

In [ ]:
import matplotlib.cm as cm

# Number of bootstrap runs used to create the files
n_bootstrap = 500

# Load data
predicted_all_runs_ssp126 = pd.read_csv("data/precomputed/predicted_traits_ssp126_all_bootstraps.csv")
predicted_all_runs_ssp370 = pd.read_csv("data/precomputed/predicted_traits_ssp370_all_bootstraps.csv")
predicted_all_runs_ssp585 = pd.read_csv("data/precomputed/predicted_traits_ssp585_all_bootstraps.csv")

# Group by bootstrap_run to get one mean and median per trait per run
plot_means_ssp126 = predicted_all_runs_ssp126.groupby("bootstrap_run")[trait_names].mean()
plot_medians_ssp126 = predicted_all_runs_ssp126.groupby("bootstrap_run")[trait_names].median()

plot_means_ssp370 = predicted_all_runs_ssp370.groupby("bootstrap_run")[trait_names].mean()
plot_medians_ssp370 = predicted_all_runs_ssp370.groupby("bootstrap_run")[trait_names].median()

plot_means_ssp585 = predicted_all_runs_ssp585.groupby("bootstrap_run")[trait_names].mean()
plot_medians_ssp585 = predicted_all_runs_ssp585.groupby("bootstrap_run")[trait_names].median()


# Calculate the 2.5th and 97.5th percentiles for CI
def compute_cis(run_level_df):
    mean_predictions = run_level_df.mean(axis=0)
    lower_ci = run_level_df.quantile(0.025, axis=0)
    upper_ci = run_level_df.quantile(0.975, axis=0)
    return mean_predictions, lower_ci, upper_ci


def plot_comparative_histograms_with_ci(summary_ssp126, summary_ssp370, summary_ssp585):
    fig, ax = plt.subplots(figsize=(12, 8), dpi=300)

    traits = summary_ssp126.index
    bar_width = 0.25
    index = range(len(traits))

    clean_labels = [clean_trait_name(trait) for trait in traits]

    cmap_126 = cm.get_cmap("Greens")
    cmap_370 = cm.get_cmap("Blues")
    cmap_585 = cm.get_cmap("Oranges")

    color_126 = mcolors.to_hex(cmap_126(0.6))
    color_370 = mcolors.to_hex(cmap_370(0.6))
    color_585 = mcolors.to_hex(cmap_585(0.6))

    ax.bar(
        index,
        summary_ssp126["mean_change"],
        bar_width,
        label="SSP126",
        color=color_126,
        alpha=0.8,
        yerr=[
            summary_ssp126["mean_change"] - summary_ssp126["lower_ci"],
            summary_ssp126["upper_ci"] - summary_ssp126["mean_change"]
        ],
        capsize=4,
        error_kw={"elinewidth": 1, "alpha": 0.8}
    )

    ax.bar(
        [i + bar_width for i in index],
        summary_ssp370["mean_change"],
        bar_width,
        label="SSP370",
        color=color_370,
        alpha=0.8,
        yerr=[
            summary_ssp370["mean_change"] - summary_ssp370["lower_ci"],
            summary_ssp370["upper_ci"] - summary_ssp370["mean_change"]
        ],
        capsize=4,
        error_kw={"elinewidth": 1, "alpha": 0.8}
    )

    ax.bar(
        [i + 2 * bar_width for i in index],
        summary_ssp585["mean_change"],
        bar_width,
        label="SSP585",
        color=color_585,
        alpha=0.8,
        yerr=[
            summary_ssp585["mean_change"] - summary_ssp585["lower_ci"],
            summary_ssp585["upper_ci"] - summary_ssp585["mean_change"]
        ],
        capsize=4,
        error_kw={"elinewidth": 1, "alpha": 0.8}
    )

    ax.set_xlabel("Traits", fontsize=14)
    ax.set_ylabel("Mean Change", fontsize=14)
    ax.set_title("", fontsize=14)

    ax.set_xticks([i + bar_width for i in index])
    ax.set_xticklabels(clean_labels, rotation=45, ha="right", fontsize=14)

    ax.tick_params(axis="y", labelsize=10)
    ax.legend(fontsize=14)

    plt.tight_layout()
    plt.show()


mean_ssp126, lower_ssp126, upper_ssp126 = compute_cis(plot_means_ssp126)
mean_ssp370, lower_ssp370, upper_ssp370 = compute_cis(plot_means_ssp370)
mean_ssp585, lower_ssp585, upper_ssp585 = compute_cis(plot_means_ssp585)

summary_ssp126 = pd.DataFrame({
    "mean_change": mean_ssp126,
    "lower_ci": lower_ssp126,
    "upper_ci": upper_ssp126
}, index=trait_names)

summary_ssp370 = pd.DataFrame({
    "mean_change": mean_ssp370,
    "lower_ci": lower_ssp370,
    "upper_ci": upper_ssp370
}, index=trait_names)

summary_ssp585 = pd.DataFrame({
    "mean_change": mean_ssp585,
    "lower_ci": lower_ssp585,
    "upper_ci": upper_ssp585
}, index=trait_names)


# Plot
plot_comparative_histograms_with_ci(
    summary_ssp126,
    summary_ssp370,
    summary_ssp585
)

# Save summary DataFrames
summary_ssp126.to_csv("output/summary_trait_change_ssp126.csv")
summary_ssp370.to_csv("output/summary_trait_change_ssp370.csv")
summary_ssp585.to_csv("output/summary_trait_change_ssp585.csv")

# Create one combined summary table with traits as rows and SSP summaries as columns
combined_trait_change_summary = pd.DataFrame(index=trait_names)

combined_trait_change_summary["SSP126_mean"] = summary_ssp126["mean_change"]
combined_trait_change_summary["SSP126_median"] = plot_medians_ssp126.median(axis=0)

combined_trait_change_summary["SSP370_mean"] = summary_ssp370["mean_change"]
combined_trait_change_summary["SSP370_median"] = plot_medians_ssp370.median(axis=0)

combined_trait_change_summary["SSP585_mean"] = summary_ssp585["mean_change"]
combined_trait_change_summary["SSP585_median"] = plot_medians_ssp585.median(axis=0)

combined_trait_change_summary = combined_trait_change_summary.reset_index()
combined_trait_change_summary = combined_trait_change_summary.rename(
    columns={"index": "trait"}
)

combined_trait_change_summary.to_csv(
    "output/summary_trait_change_all_ssps_mean_median.csv",
    index=False
)

In [ ]:
from collections import OrderedDict

trait_groups = OrderedDict({
    "Structural": [
        "Tree Height",
        "Stem Diameter",
        "Crown Height",
        "Crown Diameter",
        "Root Depth",
    ],
    "Hydraulic": [
        "Stomatal Conduct.",
        "Conduit Diam.",
    ],
    "Leaf Economics": [
        "Specific Leaf Area",
        "Leaf P",
        "Leaf Vcmax",
        "Leaf Area",
        "Leaf N",
        "Leaf Density",
        "Leaf K",
        "Leaf Thickness",
    ],
    "Woody": [
        "Bark Thickness",
        "Wood Density",
    ],
    "Tolerances": [
        "Cold",
        "Shade",
        "Water",
        "Fire Tol",
        "Myco Association",
        "Drought",
    ],
    "Reproductive": [
        "Seed Dry Mass",
    ],
})

group_colours = {
    "Structural": "#1f77b4",
    "Hydraulic": "#2ca02c",
    "Leaf Economics": "#9467bd",
    "Woody": "#e377c2",
    "Tolerances": "#bcbd22",
    "Reproductive": "#17becf",
}


def normalise_label(label):
    return (
        str(label)
        .lower()
        .replace(".", "")
        .replace("_", " ")
        .replace("-", " ")
        .strip()
    )


def get_grouped_trait_order(traits):
    """
    Match the desired group order to the actual trait names in the summaries,
    using clean_trait_name(trait) for matching.
    """
    clean_to_raw = {
        normalise_label(clean_trait_name(trait)): trait
        for trait in traits
    }

    grouped_raw_traits = []
    grouped_clean_labels = []
    grouped_trait_groups = []

    for group_name, clean_names in trait_groups.items():
        for clean_name in clean_names:
            key = normalise_label(clean_name)

            if key in clean_to_raw:
                raw_trait = clean_to_raw[key]
                grouped_raw_traits.append(raw_trait)
                grouped_clean_labels.append(clean_name)
                grouped_trait_groups.append(group_name)

    # Keep any unmatched traits at the end, so nothing silently disappears.
    unmatched_traits = [trait for trait in traits if trait not in grouped_raw_traits]

    for trait in unmatched_traits:
        grouped_raw_traits.append(trait)
        grouped_clean_labels.append(clean_trait_name(trait))
        grouped_trait_groups.append("Other")

    return grouped_raw_traits, grouped_clean_labels, grouped_trait_groups


def plot_comparative_histograms_with_ci_grouped(
    summary_ssp126_in,
    summary_ssp370_in,
    summary_ssp585_in
):
    fig, ax = plt.subplots(figsize=(14, 8), dpi=300)

    ordered_traits, clean_labels, trait_group_labels = get_grouped_trait_order(
        summary_ssp126_in.index
    )

    summary_126_grouped = summary_ssp126_in.loc[ordered_traits].copy()
    summary_370_grouped = summary_ssp370_in.loc[ordered_traits].copy()
    summary_585_grouped = summary_ssp585_in.loc[ordered_traits].copy()

    bar_width = 0.25
    index = np.arange(len(ordered_traits))

    cmap_126 = cm.get_cmap("Greens")
    cmap_370 = cm.get_cmap("Blues")
    cmap_585 = cm.get_cmap("Oranges")

    color_126 = mcolors.to_hex(cmap_126(0.6))
    color_370 = mcolors.to_hex(cmap_370(0.6))
    color_585 = mcolors.to_hex(cmap_585(0.6))

    ax.bar(
        index,
        summary_126_grouped["mean_change"],
        bar_width,
        label="SSP126",
        color=color_126,
        alpha=0.8,
        yerr=[
            summary_126_grouped["mean_change"] - summary_126_grouped["lower_ci"],
            summary_126_grouped["upper_ci"] - summary_126_grouped["mean_change"],
        ],
        capsize=4,
        error_kw={"elinewidth": 1, "alpha": 0.8},
    )

    ax.bar(
        index + bar_width,
        summary_370_grouped["mean_change"],
        bar_width,
        label="SSP370",
        color=color_370,
        alpha=0.8,
        yerr=[
            summary_370_grouped["mean_change"] - summary_370_grouped["lower_ci"],
            summary_370_grouped["upper_ci"] - summary_370_grouped["mean_change"],
        ],
        capsize=4,
        error_kw={"elinewidth": 1, "alpha": 0.8},
    )

    ax.bar(
        index + 2 * bar_width,
        summary_585_grouped["mean_change"],
        bar_width,
        label="SSP585",
        color=color_585,
        alpha=0.8,
        yerr=[
            summary_585_grouped["mean_change"] - summary_585_grouped["lower_ci"],
            summary_585_grouped["upper_ci"] - summary_585_grouped["mean_change"],
        ],
        capsize=4,
        error_kw={"elinewidth": 1, "alpha": 0.8},
    )

    ax.set_xlabel("Traits", fontsize=14)
    ax.set_ylabel("Mean Change", fontsize=14)
    ax.set_title("", fontsize=14)

    tick_positions = index + bar_width
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(clean_labels, rotation=45, ha="right", fontsize=11)

    # Colour x-axis labels by group
    for tick_label, group_name in zip(ax.get_xticklabels(), trait_group_labels):
        tick_label.set_color(group_colours.get(group_name, "black"))

    # Add group headings and horizontal lines above the bars
    y_min, y_max = ax.get_ylim()
    heading_y = y_max + (y_max - y_min) * 0.045
    line_y = y_max + (y_max - y_min) * 0.02

    for group_name in trait_groups.keys():
        group_positions = [
            i for i, label in enumerate(trait_group_labels)
            if label == group_name
        ]

        if not group_positions:
            continue

        start = min(group_positions)
        end = max(group_positions)

        group_centre = (start + end) / 2 + bar_width
        line_start = start - 0.1
        line_end = end + 2 * bar_width + 0.1

        ax.text(
            group_centre,
            heading_y,
            group_name,
            ha="center",
            va="bottom",
            fontsize=12,
            color=group_colours[group_name],
        )

        ax.hlines(
            line_y,
            line_start,
            line_end,
            colors=group_colours[group_name],
            linewidth=1.2,
        )

    ax.set_ylim(y_min, heading_y + (y_max - y_min) * 0.04)
    ax.tick_params(axis="y", labelsize=10)
    ax.legend(fontsize=12, loc = "lower right")

    plt.tight_layout()
    plt.show()

plot_comparative_histograms_with_ci_grouped(
    summary_ssp126,
    summary_ssp370,
    summary_ssp585
)

**Precipitation Leave One Group Out Cross Validation**<br>

Results can be recreated using the separate notebook file - LOGOCV.ipynb

In [ ]:
logocv_pr = pd.read_csv("data/precomputed/logocv_results_pr.csv")
logocv_pr

# Define SSP precipitation ranges relative to the baseline
ssp_scenarios_pr = ['SSP126', 'SSP370', 'SSP585']
ssp_means_pr = [49.94, 64.05, 78.62]  
lower_confidence_pr = [3.59, 26.02, 4.37] 
upper_confidence_pr = [97.5, 167.46, 185.79]  

# Plotting overall R2 and NRMSE against degrees Celsius buffering
plt.figure(figsize=(10, 6), dpi=600)

# Plot R2 and NRMSE
plt.plot(logocv_pr.buffer_size, logocv_pr.R2, marker='o', linestyle='-', color='b', label='Weighted R-squared')
plt.plot(logocv_pr.buffer_size, logocv_pr.NRMSE, marker='s', linestyle='--', color='r', label='Weighted NRMSE')

# Define colours
colors = ['lightgreen', 'orange', 'lightcoral']

# Plot SSP scenarios as shaded regions with labels and arrows
for ssp, mean, lower, upper, color in zip(ssp_scenarios_pr, ssp_means_pr, lower_confidence_pr, upper_confidence_pr, colors):
    plt.fill_betweenx([0, 1], lower, upper, color=color, alpha=0.25, edgecolor='black', linewidth=0.5, zorder=1)
    plt.annotate(ssp, 
                 xy=(mean, 0.9 - ssp_scenarios_pr.index(ssp) * 0.1), 
                 xytext=(0, -10), textcoords='offset points',
                 ha='center', va='top', fontsize=16, color='black', zorder=2)
    plt.annotate('', 
                 xy=(lower, 0.9 - ssp_scenarios_pr.index(ssp) * 0.1), 
                 xytext=(upper, 0.9 - ssp_scenarios_pr.index(ssp) * 0.1),
                 arrowprops=dict(arrowstyle='<->', color='black'), zorder=2)
    plt.plot([mean, mean], [0.9 - ssp_scenarios_pr.index(ssp) * 0.1 - 0.02, 0.9 - ssp_scenarios_pr.index(ssp) * 0.1 + 0.02],
             color='black', linestyle='-', linewidth=0.5, zorder=3)

# Annotate each point with its value for R2 and NRMSE
for i, txt in enumerate(logocv_pr.R2):
    plt.annotate(f"{txt:.2f}", (logocv_pr.buffer_size[i], logocv_pr.R2[i]),
                 textcoords="offset points", xytext=(0, 10), ha='center', va='bottom', color='b', fontsize=16)

for i, txt in enumerate(logocv_pr.NRMSE):
    plt.annotate(f"{txt:.2f}", (logocv_pr.buffer_size[i], logocv_pr.NRMSE[i]),
                 textcoords="offset points", xytext=(0, 10), ha='center', color='r', fontsize=16)

# Set axes range and tick increments
plt.xticks(np.arange(0, max(logocv_pr.buffer_size) + 40, 40), fontsize=14)
plt.ylim(0, 1.0)
plt.yticks(np.arange(0, 1.1, 0.2), fontsize=16)
plt.legend(loc='upper right', fontsize=14)

plt.xlabel('Precipitation (mm) buffering', fontsize=16)
plt.ylabel('Coefficient of determination (R2)', fontsize=16)
plt.title('', fontsize=16)
plt.grid(True)
plt.tight_layout()
plt.show()


**Temperature Leave One Group Out Cross Validation**<br>

Results can be recreated using the separate notebook file - LOGOCV.ipynb

In [ ]:
logocv_temp = pd.read_csv("data/precomputed/logocv_results_temp.csv")

# Define SSP temperature ranges relative to the baseline
ssp_scenarios = ['SSP126', 'SSP370', 'SSP585']
ssp_means = [1.59, 4.1, 5.47]  
lower_confidence = [1.0, 3.09, 4.01]  
upper_confidence = [2.63, 5.72, 7.29] 

# Plot overall R2 and NRMSE against degrees Celsius buffering
plt.figure(figsize=(10, 6), dpi=600)

# Plot R2 and NRMSE
plt.plot(logocv_temp.buffer_size, logocv_temp.R2, marker='o', linestyle='-', color='b', label='Weighted R-squared')
plt.plot(logocv_temp.buffer_size, logocv_temp.NRMSE, marker='s', linestyle='--', color='r', label='Weighted NRMSE')

# Define colours
colors = ['lightgreen', 'orange', 'lightcoral']

# Plot SSP scenarios as shaded regions with labels and arrows
for ssp, mean, lower, upper, color in zip(ssp_scenarios, ssp_means, lower_confidence, upper_confidence, colors):
    plt.fill_betweenx([0, 1], lower, upper, color=color, alpha=0.25, edgecolor='black', linewidth=0.5, zorder=1)
    plt.annotate(
        ssp, 
        xy=(mean, 0.9 - ssp_scenarios.index(ssp) * 0.1), 
        xytext=(0, -10), 
        textcoords='offset points',
        ha='center', 
        va='top', 
        fontsize=16, 
        color='black', 
        zorder=2
    )
    plt.annotate(
        '', 
        xy=(lower, 0.9 - ssp_scenarios.index(ssp) * 0.1), 
        xytext=(upper, 0.9 - ssp_scenarios.index(ssp) * 0.1),
        arrowprops=dict(arrowstyle='<->', color='black'),
        zorder=2
    )
    plt.plot(
        [mean, mean], 
        [0.9 - ssp_scenarios.index(ssp) * 0.1 - 0.02, 0.9 - ssp_scenarios.index(ssp) * 0.1 + 0.02],
        color='black', 
        linestyle='-', 
        linewidth=0.5, 
        zorder=3
    )

# Annotate each point with its value for R2 and NRMSE
for i, txt in enumerate(logocv_temp.R2):
    plt.annotate(
        f"{txt:.2f}", 
        (logocv_temp.buffer_size[i], logocv_temp.R2[i]), 
        textcoords="offset points", 
        xytext=(0, 10), 
        ha='center', 
        va='bottom', 
        color='b', 
        fontsize=16
    )

for i, txt in enumerate(logocv_temp.NRMSE):
    plt.annotate(
        f"{txt:.2f}", 
        (logocv_temp.buffer_size[i], logocv_temp.NRMSE[i]), 
        textcoords="offset points", 
        xytext=(0, 10), 
        ha='center', 
        color='r', 
        fontsize=16
    )

# Set axes range and tick increments with larger fonts
plt.xticks(np.arange(0, max(logocv_temp.buffer_size) + 1, 1), fontsize=14)
plt.ylim(0, 1.0)
plt.yticks(np.arange(0, 1.1, 0.2), fontsize=16)

# Legend placement 
plt.legend(loc='upper right', fontsize=14)
plt.xlabel('Degrees Celsius buffering', fontsize=16)
plt.ylabel('Coefficient of determination (R2)', fontsize=16)
plt.title('', fontsize=16)
plt.grid(True)
plt.tight_layout()
plt.show()